# SASRec Stage 3 Baseline Multi-Task BPI2012 Colab Train 02

This notebook is the Stage 3 follow-up version.

Main changes from `01`:
- reuse all available single-task baseline seeds (`42`, `2024`, `7`)
- compare multi-task runs with the same 3 seeds
- inspect mean/std instead of only `s42`
- show task metrics with the final metric names used in later experiments

Main comparison metric:
- `full ranking + NDCG@10`


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [ ]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"


In [ ]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [ ]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [ ]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [ ]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260601_071949
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [ ]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


## Verify Stage 3 processed file

Stage 3 next-time prediction uses the processed time-feature CSV.
This check confirms that `delta_next_seconds` already exists.


In [ ]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = [
    'delta_prev_seconds',
    'delta_start_seconds',
    'delta_next_seconds',
]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

Stage 3 baseline multi-task runs:

- backbone 1: `anchor_ml20`
- backbone 2: `refine_ml50_do035`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- time loss weight: `1.0`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison should use all 3 seeds when available: `42`, `2024`, `7`


## Check prerequisite baseline runs (all 3 seeds)


In [ ]:
from pathlib import Path

baseline_required_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

output_dir = Path(BASELINE_NDCG10_OUTPUT_DIR)
print('=' * 80)
print('Baseline NDCG@10 prerequisite runs')
for run_name in baseline_required_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10 prerequisite runs
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS


## Check planned multi-task runs


In [ ]:
planned_multitask_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]

output_dir = Path(MULTITASK_BASELINE_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 baseline multi-task runs')
for run_name in planned_multitask_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 baseline multi-task runs
multitask_anchor_ml20_s42 OK
multitask_anchor_ml20_s2024 OK
multitask_anchor_ml20_s7 OK
multitask_refine_ml50_do035_s42 OK
multitask_refine_ml50_do035_s2024 OK
multitask_refine_ml50_do035_s7 OK


## Train multi-task runs


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2/multitask_anchor_ml20_s42
epoch=1, loss=2.5741
epoch=2, loss=1.5377
epoch=3, loss=1.4230
epoch=4, loss=1.3542
epoch=5, loss=1.3119
valid [task], Top5Acc: 0.4070, Top10Acc: 0.7690, Acc: 0.0551, MacroF1: 0.0988, TimeMAE: 70134.1216, TimeRMSE: 275756.8323, TimeMedAE: 904.5147
valid [full], NDCG@5: 0.6219, HR@5: 0.7527, NDCG@10: 0.6832, HR@10: 0.9487, MRR: 0.6093
valid [sampled], NDCG@5: 0.5208, HR@5: 0.5212, NDCG@10: 0.5236, HR@10: 0.5301, MRR: 0.5364
test [task], Top5Acc: 0.0436, Top10Acc: 0.5369, Acc: 0.0141, MacroF1: 0.0150, TimeMAE: 11858.8850,

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2/multitask_anchor_ml20_s2024
epoch=1, loss=2.8037
epoch=2, loss=1.5874
epoch=3, loss=1.4188
epoch=4, loss=1.3484
epoch=5, loss=1.3018
valid [task], Top5Acc: 0.4684, Top10Acc: 0.7286, Acc: 0.0876, MacroF1: 0.1517, TimeMAE: 76872.8673, TimeRMSE: 292023.3924, TimeMedAE: 1244.3507
valid [full], NDCG@5: 0.6227, HR@5: 0.7291, NDCG@10: 0.6933, HR@10: 0.9546, MRR: 0.6205
valid [sampled], NDCG@5: 0.5291, HR@5: 0.5297, NDCG@10: 0.5344, HR@10: 0.5465, MRR: 0.5452
test [task], Top5Acc: 0.0589, Top10Acc: 0.4974, Acc: 0.0362, MacroF1: 0.0326, TimeMAE: 12709.67

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2/multitask_anchor_ml20_s7
epoch=1, loss=2.5119
epoch=2, loss=1.5382
epoch=3, loss=1.4147
epoch=4, loss=1.3497
epoch=5, loss=1.3049
valid [task], Top5Acc: 0.3614, Top10Acc: 0.7069, Acc: 0.0718, MacroF1: 0.1195, TimeMAE: 71392.0500, TimeRMSE: 288796.0744, TimeMedAE: 727.5317
valid [full], NDCG@5: 0.6093, HR@5: 0.7076, NDCG@10: 0.6958, HR@10: 0.9838, MRR: 0.6129
valid [sampled], NDCG@5: 0.5138, HR@5: 0.5159, NDCG@10: 0.5241, HR@10: 0.5483, MRR: 0.5308
test [task], Top5Acc: 0.3111, Top10Acc: 0.4077, Acc: 0.0461, MacroF1: 0.0348, TimeMAE: 12707.4961, 

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2/multitask_refine_ml50_do035_s42
epoch=1, loss=2.8483
epoch=2, loss=1.7124
epoch=3, loss=1.5800
epoch=4, loss=1.4923
epoch=5, loss=1.4453
valid [task], Top5Acc: 0.3383, Top10Acc: 0.7529, Acc: 0.0629, MacroF1: 0.0955, TimeMAE: 68418.9061, TimeRMSE: 276570.8495, TimeMedAE: 697.9265
valid [full], NDCG@5: 0.5920, HR@5: 0.7000, NDCG@10: 0.6826, HR@10: 0.9766, MRR: 0.5978
valid [sampled], NDCG@5: 0.5070, HR@5: 0.5079, NDCG@10: 0.5084, HR@10: 0.5124, MRR: 0.5223
test [task], Top5Acc: 0.0404, Top10Acc: 0.4262, Acc: 0.0250, MacroF1: 0.0223, TimeMAE: 11375

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2/multitask_refine_ml50_do035_s2024
epoch=1, loss=3.1746
epoch=2, loss=1.7878
epoch=3, loss=1.5792
epoch=4, loss=1.4786
epoch=5, loss=1.4225
valid [task], Top5Acc: 0.3538, Top10Acc: 0.6978, Acc: 0.0699, MacroF1: 0.1092, TimeMAE: 72347.3263, TimeRMSE: 292168.4166, TimeMedAE: 1140.7852
valid [full], NDCG@5: 0.6041, HR@5: 0.7009, NDCG@10: 0.6673, HR@10: 0.9053, MRR: 0.6059
valid [sampled], NDCG@5: 0.5189, HR@5: 0.5195, NDCG@10: 0.5232, HR@10: 0.5330, MRR: 0.5340
test [task], Top5Acc: 0.2883, Top10Acc: 0.5326, Acc: 0.0214, MacroF1: 0.0195, TimeMAE: 12

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2/multitask_refine_ml50_do035_s7
epoch=1, loss=2.8306
epoch=2, loss=1.7300
epoch=3, loss=1.5877
epoch=4, loss=1.5011
epoch=5, loss=1.4467
valid [task], Top5Acc: 0.3898, Top10Acc: 0.7132, Acc: 0.0637, MacroF1: 0.1066, TimeMAE: 72142.9943, TimeRMSE: 289429.2064, TimeMedAE: 978.3202
valid [full], NDCG@5: 0.5020, HR@5: 0.6528, NDCG@10: 0.5977, HR@10: 0.9491, MRR: 0.4952
valid [sampled], NDCG@5: 0.3377, HR@5: 0.3638, NDCG@10: 0.3756, HR@10: 0.4820, MRR: 0.3607
test [task], Top5Acc: 0.3551, Top10Acc: 0.6160, Acc: 0.0338, MacroF1: 0.0271, TimeMAE: 14291.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1600)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [23]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
multitask_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
multitask_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = baseline_subset['run_name'].map(
    lambda x: 'anchor_single_task' if x.startswith('anchor_ml20') else 'refine_single_task'
)

multitask_subset = multitask_df[multitask_df['run_name'].isin(multitask_runs)].copy()
multitask_subset['variant'] = multitask_subset['run_name'].map(
    lambda x: 'anchor_multi_task' if 'anchor_ml20' in x else 'refine_multi_task'
)

df_compare = pd.concat([baseline_subset, multitask_subset], ignore_index=True)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

id_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate',
    'hidden_units', 'selection_metric', 'enable_time_prediction',
    'time_prediction_target', 'time_loss_weight',
    'time_target_transform', 'time_modeling_mode',
]

metric_prefixes = (
    'best_valid_',
    'best_test_at_best_valid_',
    'last_valid_',
    'last_test_',
)

metric_cols = sorted([
    c for c in df_compare.columns
    if c.startswith(metric_prefixes)
])

display_cols = [c for c in id_cols if c in df_compare.columns] + metric_cols
df_compare[display_cols]


,run_name,seed,variant,maxlen,dropout_rate,hidden_units,selection_metric,enable_time_prediction,time_prediction_target,time_loss_weight,time_target_transform,time_modeling_mode,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_mean_rank,best_test_at_best_valid_full_median_rank,best_test_at_best_valid_full_mrr,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_num_eval_users,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_mean_rank,best_test_at_best_valid_sampled_median_rank,best_test_at_best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_num_eval_users,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_median_ae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_top1_accuracy,best_test_at_best_valid_task_top5_accuracy,best_valid_full_hr@10,best_valid_full_hr@5,best_valid_full_mean_rank,best_valid_full_median_rank,best_valid_full_mrr,best_valid_full_ndcg@10,best_valid_full_ndcg@5,best_valid_full_num_eval_users,best_valid_sampled_hr@10,best_valid_sampled_hr@5,best_valid_sampled_mean_rank,best_valid_sampled_median_rank,best_valid_sampled_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_ndcg@5,best_valid_sampled_num_eval_users,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_time_mae,best_valid_task_time_median_ae,best_valid_task_time_rmse,best_valid_task_top10_accuracy,best_valid_task_top1_accuracy,best_valid_task_top5_accuracy,last_test_full_hr@10,last_test_full_hr@5,last_test_full_mean_rank,last_test_full_median_rank,last_test_full_mrr,last_test_full_ndcg@10,last_test_full_ndcg@5,last_test_full_num_eval_users,last_test_sampled_hr@10,last_test_sampled_hr@5,last_test_sampled_mean_rank,last_test_sampled_median_rank,last_test_sampled_mrr,last_test_sampled_ndcg@10,last_test_sampled_ndcg@5,last_test_sampled_num_eval_users,last_test_task_accuracy,last_test_task_macro_f1,last_test_task_time_mae,last_test_task_time_median_ae,last_test_task_time_rmse,last_test_task_top10_accuracy,last_test_task_top1_accuracy,last_test_task_top5_accuracy,last_valid_full_hr@10,last_valid_full_hr@5,last_valid_full_mean_rank,last_valid_full_median_rank,last_valid_full_mrr,last_valid_full_ndcg@10,last_valid_full_ndcg@5,last_valid_full_num_eval_users,last_valid_sampled_hr@10,last_valid_sampled_hr@5,last_valid_sampled_mean_rank,last_valid_sampled_median_rank,last_valid_sampled_mrr,last_valid_sampled_ndcg@10,last_valid_sampled_ndcg@5,last_valid_sampled_num_eval_users,last_valid_task_accuracy,last_valid_task_macro_f1,last_valid_task_time_mae,last_valid_task_time_median_ae,last_valid_task_time_rmse,last_valid_task_top10_accuracy,last_valid_task_top1_accuracy,last_valid_task_top5_accuracy
0,multitask_anchor_ml20_s7,7,anchor_multi_task,20,0.20,50,full_valid_ndcg@10,True,delta_next_seconds,1.0,log1p,disabled,1.000000,0.937745,2.000947,1.0,0.735914,0.801466,0.782387,7389,0.444986,0.245365,17.870483,12.0,0.240381,0.265319,0.201450,7389,0.026391,0.023985,11392.935271,6.927000,66680.324584,0.595886,0.026391,0.460549,0.941655,0.854410,3.126323,1.0,0.678242,0.737962,0.711434,7370,0.613704,0.564315,17.737313,1.0,0.579719,0.575376,0.559712,7370,0.062415,0.097186,71279.254416,872.565819,287403.069435,0.646811,0.062415,0.403664,0.999864,0.936575,2.266739,2.0,0.605188,0.704814,0.685202,7363,0.278012,0.104849,19.570420,15.0,0.140627,0.140360,0.085710,7363,0.033274,0.028958,12244.773316,7.006304,67503.332274,0.592150,0.033274,0.368328,0.940043,0.844004,3.170103,1.0,0.670183,0.731218,0.701211,7372,0.603228,0.561042,18.026858,1.0,0.575956,0.570175,0.556852,7372,0.066196,0.090025,72570.272486,748.562512,281464.388283,0.623169,0.066196,0.397179
1,multitask_anchor_ml20_s42,42,

In [24]:
summary_metric_cols = sorted([
    c for c in df_compare.columns
    if c.startswith((
        'best_valid_',
        'best_test_at_best_valid_',
        'last_valid_',
        'last_test_',
    ))
])

summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_mean_rank           best_test_at_best_valid_full_median_rank      best_test_at_best_valid_full_mrr           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_num_eval_users            best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_mean_rank           best_test_at_best_valid_sampled_median_rank           best_test_at_best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_num_eval_users            best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_median_ae            best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_top1_accuracy           best_test_at_best_valid_task_top5_accuracy           best_valid_full_hr@10           best_valid_full_hr@5           best_valid_full_mean_rank           best_valid_full_median_rank      best_valid_full_mrr           best_valid_full_ndcg@10           best_valid_full_ndcg@5           best_valid_full_num_eval_users            best_valid_sampled_hr@10           best_valid_sampled_hr@5           best_valid_sampled_mean_rank            \
                                                 mean       std                              mean       std                                   mean       std                                     mean  std                             mean       std                                 mean       std                                mean       std                                        mean        std                                  mean       std                                 mean       std                                      mean       std                                        mean       std                                mean       std                                    mean       std                                   mean       std                                           mean        std                                  mean       std                                  mean       std                                  mean          std                                        mean        std                                   mean          std                                        mean       std                                       mean       std                                       mean       std                  mean       std                 mean       std                      mean       std                        mean  std                mean       std                    mean       std                   mean       std                           mean        std                     mean       std                    mean       std                         mean       std   
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

Interpretation guide:

- compare `anchor_single_task` vs `anchor_multi_task`
- compare `refine_single_task` vs `refine_multi_task`
- use `best_test_at_best_valid_full_ndcg@10` as the main Stage 3 comparison metric
- use mean/std across `42`, `2024`, `7` for final interpretation
- task metrics are shown directly as `accuracy`, `macro_f1`, `top5_accuracy`, `top10_accuracy`, `time_mae`, `time_rmse`
